# 📡 Telco Müşteri Kaybı (Churn) Analizi ve Tahmin Projesi

Bu çalışma, bir telekomünikasyon şirketinin müşteri verilerini kullanarak, hangi müşterilerin aboneliklerini iptal edeceğini (churn) tahmin etmeyi amaçlamaktadır. Müşteri kaybını önceden belirlemek, şirketin proaktif önlemler alarak müşteri bağlılığını artırmasına ve gelir kaybını azaltmasına olanak tanır. Proje kapsamında verinin temizlenmesinden, en karmaşık modellerin eğitilmesine kadar tüm süreç uçtan uca ele alınmıştır.

## 📋 Proje Akışı
1. **Veri Keşfi (EDA):** Veri setinin yapısını anlama, değişken türlerini belirleme ve ilk istatistiksel özetleri çıkarma.
2. **Veri Ön İşleme:** Eksik değerlerin bilimsel yöntemlerle doldurulması ve aykırı değerlerin baskılanması.
3. **Özellik Mühendisliği (Feature Engineering):** Ham veriden iş hedeflerine yönelik yeni ve anlamlı değişkenlerin türetilmesi.
4. **Model Geliştirme:** Makine öğrenmesi modellerinin eğitilmesi ve en iyi parametrelerin Optuna kütüphanesi ile bulunması.
5. **Model Değerlendirme:** Modelin başarısını ölçmek için karmaşıklık matrisi ve önem sıralaması gibi metriklerin analizi.
6. **Sonuç ve Sunum:** Elde edilen bulguların iş dünyası için stratejik önerilere dönüştürülmesi.

## 🛠️ 1. Kütüphaneler ve Ayarlar
Analiz sürecinde veri manipülasyonu için `pandas`, görselleştirme için `seaborn` ve makine öğrenmesi algoritmaları için `scikit-learn`, `XGBoost` gibi kütüphaneler kullanılmaktadır. Ayrıca modellerin başarısını optimize etmek amacıyla `optuna` kütüphanesi projeye dahil edilmiştir.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna
from feature_engineering import create_features

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 📊 2. Veri Yükleme ve İlk Bakış
Ham veri seti sisteme yüklenerek sütun başlıkları ve veri tipleri incelenir. Bu aşama, verideki hatalı formatları (örneğin sayısal olması gereken alanların metin olarak görünmesi) tespit etmek için kritik öneme sahiptir.

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"Veri Seti Boyutu: {df.shape}")
df.head()

### Veri Tipleri ve Hatalı Formatların Düzeltilmesi
`TotalCharges` sütununda bulunan boşluk karakterleri verinin sayısal olarak işlenmesini engellemektedir. Bu karakterler önce `NaN` değerine dönüştürülmekte, ardından tüm sütun ondalıklı sayı (float) tipine çevrilmektedir.

In [ ]:
# TotalCharges numeric olmalı ama boşluk içeren stringler var
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.info()

## 🧹 3. Veri Ön İşleme (Preprocessing)

### Eksik Değerlerin KNN ile Doldurulması
Veri setindeki eksik değerler rastgele silinmek yerine, en yakın komşuluk (KNN) algoritması kullanılarak diğer benzer müşterilerin verileriyle doldurulur. Bu sayede veri kaybı önlenirken, verinin doğal yapısı ve dağılımı korunmuş olur.

In [ ]:
print(f"Doldurma öncesi eksik TotalCharges: {df['TotalCharges'].isnull().sum()}")
imputer = KNNImputer(n_neighbors=5)
df[['tenure', 'MonthlyCharges', 'TotalCharges']] = imputer.fit_transform(df[['tenure', 'MonthlyCharges', 'TotalCharges']])
print(f"Doldurma sonrası eksik TotalCharges: {df['TotalCharges'].isnull().sum()}")

### Aykırı Değer (Outlier) Analizi ve Baskılama
İstatistiksel olarak uç noktalarda bulunan (aykırı) değerler, modelin öğrenme sürecini olumsuz etkileyebilir. IQR (Çeyrekler Arası Aralık) yöntemi ile tespit edilen bu değerler, alt ve üst sınır değerlerine çekilerek (capping) verinin gürültüden arındırılması sağlanır.

In [ ]:
def handle_outliers(df, columns):
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df[col] = np.clip(df[col], lower_bound, upper_bound)
    return df

df = handle_outliers(df, ['tenure', 'MonthlyCharges', 'TotalCharges'])

## ⚙️ 4. Özellik Mühendisliği (Feature Engineering)
Mevcut verileri birleştirerek veya oranlayarak modelin daha iyi anlam çıkarabileceği yeni sütunlar oluşturuyoruz. Örneğin, müşterinin ortalama aylık ödemesi veya kullandığı toplam servis sayısı gibi bilgiler churn tahmininde oldukça etkilidir.

In [ ]:
df = create_features(df)
df.head()

## 🔢 5. Encoding ve Ölçeklendirme
Algoritmaların veriyi işleyebilmesi için metinsel (kategorik) ifadeler `Label Encoding` ve `One-Hot Encoding` yöntemleriyle sayısallaştırılır. Ardından, büyük değerlerin küçük değerleri baskılamaması için tüm sayısal değişkenler `StandardScaler` ile standart bir aralığa getirilir.

In [ ]:
# customerID gereksiz
if 'customerID' in df.columns:
    df.drop('customerID', axis=1, inplace=True)

# Binary Encoding
le = LabelEncoder()
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# One-Hot Encoding
multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
              'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 
              'Contract', 'PaymentMethod']
df = pd.get_dummies(df, columns=multi_cols, drop_first=True, dtype=int)

# Scaling
scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharge', 'ChargeRatio', 'tenure_x_Monthly']
df[num_cols] = scaler.fit_transform(df[num_cols])

## 🤖 6. Model Geliştirme ve Hiperparametre Optimizasyonu

### Verinin Bölünmesi ve Strateji
Veri seti, modelin hiç görmediği veriler üzerindeki başarısını test etmek amacıyla eğitim (%80) ve test (%20) olarak ikiye ayrılır. Eğitim sırasında sınıflar arasındaki dengesizliği korumak için `stratify` parametresi kullanılarak churn oranının her iki sette de aynı kalması sağlanır.

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Eğitim seti boyutu: {X_train.shape}")

### Optuna ile Akıllı Parametre Arama
Modelin en yüksek doğruluğa ulaşması için gereken derinlik, öğrenme oranı gibi kritik ayarlar `Optuna` kütüphanesi tarafından otomatik olarak aranır. Bu süreç, manuel deneme yanılma yöntemine göre çok daha hızlı ve bilimsel sonuçlar üretir.

In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42,
        'use_label_encoder': False,
        'eval_metric': 'logloss'
    }
    model = XGBClassifier(**params)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='f1').mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

print(f"En iyi F1 Skoru: {study.best_value}")

## 📈 7. Model Değerlendirme ve Görselleştirme
Eğitilen modelin gerçek hayattaki performansı, test verileri kullanılarak ölçülür. Başarı değerlendirmesinde sadece genel doğruluk (Accuracy) değil, churn olan müşterileri ne kadar isabetle bulabildiğimizi gösteren Recall ve F1-Score metriklerine odaklanılır.

In [ ]:
best_params = study.best_params
xgb_model = XGBClassifier(**best_params)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))

### Karmaşıklık Matrisi (Confusion Matrix)
Bu grafik, modelin kaç müşteriyi doğru tahmin ettiğini ve hangi sınıflarda (Churn/No Churn) hata yaptığını detaylı olarak gösterir. Özellikle "kaçırılan" müşterilerin (False Negative) sayısını azaltmak stratejik bir önceliktir.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title('Karmaşıklık Matrisi')
plt.show()

### Özellik Önem Sıralaması (Feature Importance)
Modelin karar verirken hangi faktörlere daha çok güvendiğini bu grafik üzerinden görebiliriz. Bu analiz, şirketin hangi alanlarda (örneğin sözleşme tipi veya internet hızı) iyileştirme yapması gerektiğine dair net bir yol haritası sunar.

In [ ]:
feat_importances = pd.Series(xgb_model.feature_importances_, index=X.columns)
feat_importances.nlargest(10).plot(kind='barh', color='skyblue')
plt.title('Tahmin Üzerinde En Etkili 10 Özellik')
plt.show()

## 🎯 8. Sonuç ve Öneriler

1. **Model Başarısı:** Geliştirilen model, churn olasılığını yüksek bir güvenle tahmin edebilmekte ve riskli grupları net bir şekilde ayrıştırabilmektedir.
2. **Kritik Bulgular:** Analiz sonuçları, özellikle aylık sözleşme kullanan ve yeni abonelik başlatan müşterilerin en riskli grup olduğunu göstermektedir.
3. **Önerilen Aksiyon:** Şirket yönetimi, bu riskli gruplara yönelik sadakat programları veya uzun dönemli sözleşme indirimleri uygulayarak müşteri kaybını minimize edebilir.